In [168]:
import pandas as pd
from sqlalchemy import create_engine

In [169]:
jsonPath = "dados/farmago.json"

In [170]:
try:
    df = pd.read_json(jsonPath, encoding="utf-8")
    print("Data Frame criado a partir do JSON.")
except Exception as e:
    print("Erro ao criar o Data Frame a partir do JSON")

Data Frame criado a partir do JSON.


In [171]:
dfFarmaGo = pd.json_normalize(
    df["medicamentos"],
    record_path="farmacias",
    meta=["nome", "categoria"],
    meta_prefix="medicamento_"
)
dfFarmaGo

,nome,preco,distancia_km,estoque,medicamento_nome,medicamento_categoria
0,Farmacia Central,12.9,1.2,25,Dipirona 500mg,Analgesico
0,Drogaria Brasil,10.5,3.8,8,Dipirona 500mg,Analgesico
0,Farmacia Popular,11.9,2.1,15,Dipirona 500mg,Analgesico
1,Farmacia Central,15.9,1.2,30,Paracetamol 750mg,Analgesico
1,Drogaria Brasil,13.5,3.8,12,Paracetamol 750mg,Analgesico
1,Farmacia Popular,14.2,2.1,4,Paracetamol 750mg,Analgesico
2,Farmacia Central,18.9,1.2,7,Ibuprofeno 600mg,Anti-inflamatorio
2,Drogaria Brasil,16.4,3.8,20,Ibuprofeno 600mg,Anti-inflamatorio
2,Farmacia Popular,17.9,2.1,10,Ibuprofeno 600mg,Anti-inflamatorio


In [190]:
averagePricePerProduct = dfFarmaGo.groupby("medicamento_nome")["preco"].mean().reset_index()
highestAverage = averagePricePerProduct.loc[averagePricePerProduct["preco"].idxmax(), "medicamento_nome"]
print(f"Highest Average: {highestAverage}")

Highest Average: Ibuprofeno 600mg


In [175]:
lessThan15 = dfFarmaGo[dfFarmaGo["preco"] < 15]
lessThan15

,nome,preco,distancia_km,estoque,medicamento_nome,medicamento_categoria
0,Farmacia Central,12.9,1.2,25,Dipirona 500mg,Analgesico
0,Drogaria Brasil,10.5,3.8,8,Dipirona 500mg,Analgesico
0,Farmacia Popular,11.9,2.1,15,Dipirona 500mg,Analgesico
1,Drogaria Brasil,13.5,3.8,12,Paracetamol 750mg,Analgesico
1,Farmacia Popular,14.2,2.1,4,Paracetamol 750mg,Analgesico


In [176]:
largestInventory = dfFarmaGo.groupby("medicamento_nome")["estoque"].sum() 
largestInventory.idxmax()

'Dipirona 500mg'

In [177]:
dfCheaper = dfFarmaGo.groupby("medicamento_nome")["preco"].min().reset_index()

In [178]:
dfMoreExpensive = dfFarmaGo.groupby("medicamento_nome")["preco"].max().reset_index()

In [179]:
dfDifference = pd.DataFrame()

In [180]:
dfDifference["medicamento_nome"] = dfCheaper["medicamento_nome"]
dfDifference["difference"] = dfMoreExpensive["preco"] - dfCheaper["preco"]

In [181]:
dfDifference[dfDifference["difference"] == dfDifference["difference"].max()]

,medicamento_nome,difference
1,Ibuprofeno 600mg,2.5


In [182]:
lessThanAverageDistance = dfFarmaGo[dfFarmaGo["distancia_km"] < dfFarmaGo["distancia_km"].mean()]
lessThanAverageDistance.head()

,nome,preco,distancia_km,estoque,medicamento_nome,medicamento_categoria
0,Farmacia Central,12.9,1.2,25,Dipirona 500mg,Analgesico
0,Farmacia Popular,11.9,2.1,15,Dipirona 500mg,Analgesico
1,Farmacia Central,15.9,1.2,30,Paracetamol 750mg,Analgesico
1,Farmacia Popular,14.2,2.1,4,Paracetamol 750mg,Analgesico
2,Farmacia Central,18.9,1.2,7,Ibuprofeno 600mg,Anti-inflamatorio


In [183]:
dfCloserPharmacies = lessThanAverageDistance[["nome", "distancia_km"]].drop_duplicates().reset_index(drop=True)
dfCloserPharmacies

,nome,distancia_km
0,Farmacia Central,1.2
1,Farmacia Popular,2.1


In [184]:
averagePricePerPharmacy = dfFarmaGo.groupby("nome")["preco"].mean().reset_index()

In [185]:
dfCheapPharmacies = averagePricePerPharmacy[averagePricePerPharmacy["preco"] < dfFarmaGo["preco"].mean()].reset_index(drop=True)
dfCheapPharmacies

,nome,preco
0,Drogaria Brasil,13.466667
1,Farmacia Popular,14.666667


In [186]:
betterPharmacies = [
    farmacia for farmacia in dfCheapPharmacies["nome"] if farmacia in dfCloserPharmacies["nome"].values
]
print(betterPharmacies)

['Farmacia Popular']


In [187]:
engine = create_engine("sqlite:///farmago.db")

try:
    dfFarmaGo.to_sql('medicamentos', con=engine, if_exists='replace', index=False)
    print("Dados exportados com sucesso para a tabela de medicamentos")
except Exception as e:
    print(f"Ocorreu um erro ao exportar os dados {e}")

Dados exportados com sucesso para a tabela de medicamentos
